In [5]:
import pandas as pd
moca = pd.read_csv('cf_data/mocadb.csv', dtype={'GaiaDR3_ID': 'Int64'})
print(moca.shape)
print(f'Duplicate GaiaDR3_IDs: {moca["GaiaDR3_ID"].duplicated().sum()}')
print(f'Missing GaiaDR3_ID: {moca["GaiaDR3_ID"].isna().sum()}')

(79853, 24)
Duplicate GaiaDR3_IDs: 253
Missing GaiaDR3_ID: 14


In [8]:
missing_moca = moca[moca['GaiaDR3_ID'].isna()]
print(missing_moca[['moca_oid', 'designation', 'twomass_id']].to_string())

       moca_oid                   designation        twomass_id
40112   4253809   Gaia DR2 452525936227352832  02313037+5333032
40113   4253971   Gaia DR2 453111220010221184  02432739+5325066
40114   4253705   Gaia DR2 452170068116898304  02321719+5228079
40115   4252108   Gaia DR2 440458108916178560  02541207+5112367
40116   4254452   Gaia DR2 455485679790245760  02233025+5337343
40117   4254575   Gaia DR2 456265615789196928  02105261+5455382
40118   4254223   Gaia DR2 453783021615354624  02402711+5327394
40119   4254292   Gaia DR2 454152079565744384  02334186+5450389
66201   4216773  Gaia DR2 2018005712801150976  19265148+2054549
66202   4253330  Gaia DR2 4517028345137608448  19185191+2123281
66203   4216826  Gaia DR2 2019072612707299712  19233380+2209126
66206   4253253  Gaia DR2 4514941231566651648  19182027+1734207
77459    141322  Gaia DR2 5530606606553877632  07504235-4638328
78236     15470       2MASS J12281366-4316388  12281366-4316388


In [10]:
from astroquery.gaia import Gaia

dr2_ids = []
for _, row in missing_moca.iterrows():
    if 'Gaia DR2' in str(row['designation']):
        dr2_id = int(row['designation'].replace('Gaia DR2', '').strip())
        dr2_ids.append(dr2_id)

print(f'DR2 IDs to query: {len(dr2_ids)}')

ids_str = ', '.join(str(i) for i in dr2_ids)
query = f"""
SELECT dr2_source_id, dr3_source_id
FROM gaiadr3.dr2_neighbourhood
WHERE dr2_source_id IN ({ids_str})
"""
result = Gaia.launch_job(query).get_results().to_pandas()
print(result)

The archive is unstable and may perform below expectations. Please avoid launching intense Python query showers. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the recent infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive
DR2 IDs to query: 13
          dr2_source_id        dr3_source_id
0    452525936227352832   452525936227352832
1   4517028345137608448  4517028345159224192
2    453783021615354624   453783021615354624
3    453111220010221184   453111220010221184
4    454152079565744384   454152079565744384
5    452170068116898304   452170068116898304
6    455485679790245760   455485679790245760
7   4517028345137608448  4517028345137608448
8   5530606606553877632  5530606610856277376
9   2019072612707299712  2019072612707299712
10  5530606606553877632  5530606610851820160
11   456265615789196928   456265615789196928
12  2018005712801150976  201800571

In [15]:
ids = moca.dropna(subset=['GaiaDR3_ID'])['GaiaDR3_ID'].drop_duplicates().tolist()
print(f'Unique IDs to query: {len(ids)}')

Unique IDs to query: 79611


In [16]:
from tqdm.notebook import tqdm

batch_size = 1000
all_results = []

batches = [ids[i:i+batch_size] for i in range(0, len(ids), batch_size)]

for batch in tqdm(batches, desc='Querying Gaia'):
    ids_str = ', '.join(str(i) for i in batch)
    query = f"""
    SELECT source_id, bp_rp, ebpminrp_gspphot, ruwe, parallax
    FROM gaiadr3.gaia_source
    WHERE source_id IN ({ids_str})
    """
    try:
        result = Gaia.launch_job(query).get_results().to_pandas()
        all_results.append(result)
    except:
        print(f'Batch failed, skipping')

gaia_results_moca = pd.concat(all_results).reset_index(drop=True)
gaia_results_moca['source_id'] = gaia_results_moca['source_id'].astype('Int64')
print(f'Total Gaia results: {len(gaia_results_moca)} rows')

Querying Gaia:   0%|          | 0/80 [00:00<?, ?it/s]

Total Gaia results: 79611 rows


In [19]:
moca = moca.merge(
    gaia_results_moca.rename(columns={'source_id': 'GaiaDR3_ID'}),
    on='GaiaDR3_ID',
    how='left'
)

moca['bprp0'] = moca['bp_rp'] - moca['ebpminrp_gspphot']

print(f'Merged: {len(moca)} rows')
print(f'Missing bprp0: {moca["bprp0"].isna().sum()}')

Merged: 79853 rows
Missing bprp0: 45544


In [20]:
moca

,moca_oid,GaiaDR3_ID,twomass_id,moca_aid,moca_mtid,designation,ra,dec,spectral_type,banyan_prob,...,age_myr_unc_neg,k_m,k_cmsig,k_msigcom,k_snr,bp_rp,ebpminrp_gspphot,ruwe,parallax,bprp0
0,1878244,414896422071032576,00531960+4949597,ALES1,CM,Gaia DR3 414896422071032576,13.331774,49.833230,(M0.0),98.7492,...,NaN,14.132,0.0563,0.0570,20.12,2.119610,0.2034,1.161392,1.400698,1.916210
1,1878109,402886387842581376,00534726+4946516,ALES1,CM,Gaia DR3 402886387842581376,13.446953,49.781013,(M0.0),35.3649,...,NaN,14.416,0.0760,0.0765,15.49,2.040987,0.2125,1.354186,1.528976,1.828487
2,1878083,402696000532908032,01055822+4939462,ALES1,CM,Gaia DR3 402696000532908032,16.492647,49.662811,(M0.2),94.4771,...,NaN,14.481,0.0775,0.0780,13.62,2.086103,0.4270,1.040434,1.461873,1.659103
3,1878190,414337728428612736,00501046+4825537,ALES1,CM,Gaia DR3 414337728428612736,12.543626,48.431545,(M0.3),NaN,...,NaN,14.912,0.1032,0.1036,10.15,2.285215,0.4240,1.025568,1.295034,1.861215
4,1878215,414511932302683136,00515953+4929034,ALES1,CM,Gaia DR3 414511932302683136,12.998100,49.484276,(M0.4),96.9182,...,NaN,14.875,0.1071,0.1074,10.51,2.209208,0.3132,1.026454,1.445765,1.896008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79848,30565,178958956375389184,04352204+3925264,THEIA1347,CM,Gaia DR2 178958956375389184,68.842149,39.423736,(M2.7),NaN,...,387.6,13.552,0.0419,0.0428,29.90,2.748308,0.4851,0.977718,5.388366,2.263208
79849,30586,179161919350169856,04273420+3806241,THEIA1347,CM,Gaia DR2 179161919350169856,66.892838,38.106425,(M6.0),NaN,...,387.6,13.240,0.0403,0.0412,47.36,2.897964,0.4153,0.901838,7.230973,2.482664
79850,30596,179307742079823616,04212084+3813408,THEIA1347,CM,Gaia DR2 179307742079823616,65.337206,38.227630,(M3.4),NaN,...,387.6,12.185,0.0228,0.0244,134.10,2.749601,0.3482,1.055583,9.310546,2.401401
79851,30602,179641478218819328,04305305+3935194,THEIA1347,CM,Gaia DR2 179641478218819328,67.721371,39.588452,(M6.0),NaN,...,387.6,12.900,0.0300,0.0312,52.65,2.758438,0.5186,1.019775,5.578903,2.239838


In [21]:
missing_bprp0 = moca[moca['bprp0'].isna()]
print(missing_bprp0['parallax_mas'].describe())

count    45526.000000
mean         0.859162
std          1.103699
min         -1.850605
25%          0.233042
50%          0.595753
75%          1.100921
max         24.280000
Name: parallax_mas, dtype: float64


In [22]:
moca['bprp0'] = moca['bp_rp'] - moca['ebpminrp_gspphot']

print(f'Stars with valid bprp0: {moca["bprp0"].notna().sum()}')
print(f'Stars missing bprp0: {moca["bprp0"].isna().sum()}')

moca.to_csv('cf_data/mocadb.csv', index=False)
print('Saved!')

Stars with valid bprp0: 34309
Stars missing bprp0: 45544
Saved!


In [24]:
query = """
SELECT source_id
FROM gaiadr3.tmass_psc_xsc_best_neighbour
WHERE original_ext_source_id = '12281366-4316388'
"""
result = Gaia.launch_job(query).get_results().to_pandas()
print(result)

             source_id
0  6145505739903448192
1  6145505739906596864


In [28]:
candidate_ids = [6145505739903448192, 6145505739906596864]
ids_str = ', '.join(str(i) for i in candidate_ids)
query2 = f"""
SELECT source_id, bp_rp, ebpminrp_gspphot, ruwe, parallax
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""
gaia_check = Gaia.launch_job(query2).get_results().to_pandas()
print(gaia_check)

             source_id     bp_rp  ebpminrp_gspphot      ruwe   parallax
0  6145505739903448192       NaN               NaN  0.966764  16.974497
1  6145505739906596864  3.126579               NaN  1.307865  17.122478


In [31]:
moca

,moca_oid,GaiaDR3_ID,twomass_id,moca_aid,moca_mtid,designation,ra,dec,spectral_type,banyan_prob,...,age_myr_unc_neg,k_m,k_cmsig,k_msigcom,k_snr,bp_rp,ebpminrp_gspphot,ruwe,parallax,bprp0
0,1878244,414896422071032576,00531960+4949597,ALES1,CM,Gaia DR3 414896422071032576,13.331774,49.833230,(M0.0),98.7492,...,NaN,14.132,0.0563,0.0570,20.12,2.119610,0.2034,1.161392,1.400698,1.916210
1,1878109,402886387842581376,00534726+4946516,ALES1,CM,Gaia DR3 402886387842581376,13.446953,49.781013,(M0.0),35.3649,...,NaN,14.416,0.0760,0.0765,15.49,2.040987,0.2125,1.354186,1.528976,1.828487
2,1878083,402696000532908032,01055822+4939462,ALES1,CM,Gaia DR3 402696000532908032,16.492647,49.662811,(M0.2),94.4771,...,NaN,14.481,0.0775,0.0780,13.62,2.086103,0.4270,1.040434,1.461873,1.659103
3,1878190,414337728428612736,00501046+4825537,ALES1,CM,Gaia DR3 414337728428612736,12.543626,48.431545,(M0.3),NaN,...,NaN,14.912,0.1032,0.1036,10.15,2.285215,0.4240,1.025568,1.295034,1.861215
4,1878215,414511932302683136,00515953+4929034,ALES1,CM,Gaia DR3 414511932302683136,12.998100,49.484276,(M0.4),96.9182,...,NaN,14.875,0.1071,0.1074,10.51,2.209208,0.3132,1.026454,1.445765,1.896008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79848,30565,178958956375389184,04352204+3925264,THEIA1347,CM,Gaia DR2 178958956375389184,68.842149,39.423736,(M2.7),NaN,...,387.6,13.552,0.0419,0.0428,29.90,2.748308,0.4851,0.977718,5.388366,2.263208
79849,30586,179161919350169856,04273420+3806241,THEIA1347,CM,Gaia DR2 179161919350169856,66.892838,38.106425,(M6.0),NaN,...,387.6,13.240,0.0403,0.0412,47.36,2.897964,0.4153,0.901838,7.230973,2.482664
79850,30596,179307742079823616,04212084+3813408,THEIA1347,CM,Gaia DR2 179307742079823616,65.337206,38.227630,(M3.4),NaN,...,387.6,12.185,0.0228,0.0244,134.10,2.749601,0.3482,1.055583,9.310546,2.401401
79851,30602,179641478218819328,04305305+3935194,THEIA1347,CM,Gaia DR2 179641478218819328,67.721371,39.588452,(M6.0),NaN,...,387.6,12.900,0.0300,0.0312,52.65,2.758438,0.5186,1.019775,5.578903,2.239838


In [32]:
print(moca[moca['bprp0'].isna()][['bp_rp', 'ebpminrp_gspphot']].isna().sum())

bp_rp                  11
ebpminrp_gspphot    45544
dtype: int64


In [3]:
import pandas as pd

moca = pd.read_csv('cf_data/mocadb.csv', dtype={'GaiaDR3_ID': 'Int64'})

In [9]:
moca['prot_days'].notna().sum()

np.int64(645)